# ARC-AGI-2 · sorokin-4B TTT (Phase A) → gpt-oss-120b verified program synthesis (Phase B) — v2

**Roles.** Phase A (4B, one L4 per worker, Unsloth LoRA TTT + DFS + augmentation scoring) is the score engine and is untouched except for scheduling knobs. Phase B (gpt-oss-120b, vLLM 0.11.0 TP=4, Marlin MXFP4 on all four L4s) runs *after* it and may only add evidence-backed candidates: programs that reproduce **every** training pair. Hints flow 4B → 120B (the 4B has a 16-token vocab and cannot read); a verified program that agrees with one of the 4B's own candidates is the strongest signal and is the only thing allowed to displace a solid attempt_2. attempt_1 from the 4B is never touched.

**Modes** (`ARC_MODE`): `hybrid` (default) · `a_only` (== original notebook, byte-for-byte behaviour) · `b_only` (120B alone; diagnostic — expect low single digits).

**Inputs to attach**: competition data · `sorokin/qwen3_4b_grids15_sft139` (Models) · `danielhanchen/gpt-oss-120b` (Models) · a dataset with the Phase B wheelhouse (`build_wheelhouse_b.py` → 124 wheels, ~1 GB). Environment: **pin to docker 31090** (Python 3.11 / torch 2.8.0+cu128).

## Settings (all env vars, defaults shown are the recommended ones)

| Area | Knob | Default | Why |
|---|---|---|---|
| Budget | `ARC_PHASE_B_RESERVE_MIN` | 120 | 120B window = install 3 + init ≤45 + generation + merge; the 4B keeps ~10 h |
| Phase A | `ARC_ORDER` | `sjf` (hybrid) | small tasks first: highest solve-rate per GPU-minute; long tasks are the ones the cap trims |
| Phase A | `ARC_FAIR_SHARE` / `ARC_MIN_CAP_S` | 1 / 600 | per-task cap = remaining time ÷ remaining tasks per worker, floor 600 s → the whole queue gets TTT + a decode batch instead of the tail getting nothing |
| Phase A | `ARC_MIN_TASK_S` | 480 | never start a task that cannot finish TTT + decode |
| Phase A | `ARC_TTT_AUG_N` / `ARC_TASK_CAP_S` | 16 / 1200 | original values; 8 halves TTT time (small quality loss) — only if you add a second 4B |
| Phase B | `ARC_B_GPU_UTIL` / `ARC_B_MAX_NUM_SEQS` | 0.88 / 32 | ~16 GB weights per L4 + KV; small sampler/graph footprint (the 27B OOM lesson); eager retry at 0.92/16 |
| Phase B | `ARC_B_MAX_MODEL_LEN` / `ARC_B_MAX_BATCHED_TOKENS` | 16384 / 4096 | prompts ≤ ~5k tokens, completions ≤ 6k |
| Phase B | `ARC_B_K1`/`K2`/`K4`, `ARC_B_EFFORT1`/`EFFORT4` | 6/2/4, medium/high | fresh → repair → direct → high-effort rounds, each time-checked |
| Phase B | `ARC_B_HINTS` | 1 | 4B proposals shown as fallible hints; lookup-table programs are rejected |
| Merge | `ARC_MERGE_POLICY` | `agree` | fill empty slots; replace attempt_2 only for single-beam att2 or 4B↔120B agreement |

Safety: `submission.json` is written right after Phase A; every Phase B failure mode (not attached, install fails, engine OOM, timeout) leaves it untouched.


In [ ]:
import os, sys, json, time, glob
from pathlib import Path

# ───────────────────────────── time budget ─────────────────────────────
T0 = time.time()
HARD_LIMIT_H = float(os.getenv("ARC_HARD_LIMIT_H", "12"))
global_end_time = T0 + HARD_LIMIT_H * 3600 - 600          # identical to the original notebook
HARD_END = global_end_time

# ───────────────────────────── mode ────────────────────────────────────
# hybrid : Phase A (sorokin 4B TTT, 4 x L4 workers) -> Phase B (gpt-oss-120b, vLLM TP=4) gap-fill
# a_only : original notebook behaviour (Phase A gets the whole budget)
# b_only : gpt-oss-120b program synthesis on every task (experiment / diagnosis mode)
ARC_MODE = os.getenv("ARC_MODE", "hybrid").lower()
assert ARC_MODE in ("hybrid", "a_only", "b_only"), ARC_MODE

# Minutes reserved at the end for Phase B (install ~3 + engine init <=45 + generation + merge).
# Phase B only ever spends what Phase A leaves over; the 4B is the better use of GPU-hours.
PHASE_B_RESERVE_MIN = float(os.getenv("ARC_PHASE_B_RESERVE_MIN", "120"))
PHASE_B_MERGE_BUFFER_S = 8 * 60                            # hard-kill Phase B this long before HARD_END

if ARC_MODE == "a_only":
    PHASE_A_END = HARD_END
elif ARC_MODE == "b_only":
    PHASE_A_END = T0
else:
    PHASE_A_END = HARD_END - PHASE_B_RESERVE_MIN * 60
PHASE_B_KILL_AT = HARD_END - PHASE_B_MERGE_BUFFER_S

# ───────────────────────────── paths ───────────────────────────────────
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
COMP_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
TEST_PATH = f"{COMP_DIR}/arc-agi_test_challenges.json" if RERUN else f"{COMP_DIR}/arc-agi_evaluation_challenges.json"
SOL_PATH = None if RERUN else f"{COMP_DIR}/arc-agi_evaluation_solutions.json"

GPT_OSS_DIR = os.getenv("ARC_GPT_OSS_PATH", "/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1")
if not Path(GPT_OSS_DIR, "config.json").exists():
    for cfg in sorted(Path("/kaggle/input").glob("**/config.json")):
        try:
            if json.loads(cfg.read_text()).get("model_type") == "gpt_oss":
                GPT_OSS_DIR = str(cfg.parent); break
        except Exception:
            pass

WHEELHOUSE = os.getenv("ARC_WHEELHOUSE_B", "")
if not WHEELHOUSE or not glob.glob(f"{WHEELHOUSE}/vllm-*.whl"):
    hits = glob.glob("/kaggle/input/**/vllm-0.11.0*.whl", recursive=True)
    WHEELHOUSE = str(Path(hits[0]).parent) if hits else ""

WORK = "/kaggle/working"
CFG = dict(
    t0=T0, hard_end=HARD_END, mode=ARC_MODE, rerun=RERUN,
    phase_a_end=PHASE_A_END, phase_b_kill_at=PHASE_B_KILL_AT,
    test_path=TEST_PATH, sol_path=SOL_PATH,
    gpt_oss_dir=GPT_OSS_DIR, wheelhouse=WHEELHOUSE, venv_b=f"{WORK}/vllm_env",
    inference_outputs="/kaggle/inference_outputs",
    phase_a_stats=f"{WORK}/phase_a_stats.json",
    phase_b_results=f"{WORK}/phase_b/results.json",
    phase_b_stage=f"{WORK}/phase_b/stage.txt",
    dev_tasks=os.getenv("ARC_DEV_TASKS", "0934a4d8,36a08778,981571dc,aa4ec2a5"),
    python=sys.version.split()[0],
)
os.makedirs(f"{WORK}/phase_b", exist_ok=True)
json.dump(CFG, open(f"{WORK}/arc_run_config.json", "w"), indent=1)

print(f"python {CFG['python']} | mode={ARC_MODE} | rerun={RERUN}")
print(f"Phase A budget: {(PHASE_A_END - T0)/3600:.2f} h | Phase B window: {(PHASE_B_KILL_AT - PHASE_A_END)/60:.0f} min")
print(f"gpt-oss dir  : {GPT_OSS_DIR} ({'found' if Path(GPT_OSS_DIR,'config.json').exists() else 'MISSING'})")
print(f"wheelhouse B : {WHEELHOUSE or 'MISSING (Phase B will be skipped)'}")


In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import json
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)


def resolve_model_dir():
    """Locate the sorokin grid-SFT Qwen3-4B (16-token ARC vocab) used by this TTT pipeline.

    Order: ARC_MODEL_PATH env -> known mount -> scan /kaggle/input for a config.json with a
    tiny vocab (the ARC cut tokenizer, vocab_size <= 32). The gpt-oss-120b checkpoint is
    deliberately NOT a candidate here: it cannot be loaded by this single-GPU Unsloth/LoRA
    path (63 GB MXFP4 MoE, 200k-token harmony vocab) - it is served by Phase B instead.
    """
    import json
    from pathlib import Path

    candidates = []
    env_path = os.getenv("ARC_MODEL_PATH")
    if env_path:
        candidates.append(env_path)
    candidates += [
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/default/1",
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/default/1",
    ]
    for candidate in candidates:
        path = Path(candidate)
        if (path / "config.json").exists():
            print(f"Using model dir: {path}")
            return str(path)

    for config in sorted(Path("/kaggle/input").glob("**/config.json")):
        try:
            cfg = json.loads(config.read_text())
        except Exception:
            continue
        if cfg.get("model_type") == "gpt_oss":
            continue
        if int(cfg.get("vocab_size", 10**9)) <= 32:
            print(f"Using discovered ARC-vocab model dir: {config.parent}")
            return str(config.parent)

    roots = []
    try:
        roots = [p.name for p in Path("/kaggle/input").iterdir()]
    except Exception:
        pass
    raise FileNotFoundError(
        "Could not locate the 16-vocab Qwen grid model (sorokin/qwen3_4b_grids15_sft139). "
        "Attach it via Add Input -> Models, or set ARC_MODEL_PATH. "
        f"input_roots={roots[:40]}"
    )


def stable_seed_for_key(key, offset=0):
    base = sum((i + 1) * ord(ch) for i, ch in enumerate(str(key)))
    return (base + int(offset)) % (1024 ** 2)


ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "ÄŠ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # ðŸ”§ KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    # Knobs (defaults reproduce the original notebook exactly)
    TTT_AUG_N = int(os.getenv("ARC_TTT_AUG_N", "16"))      # 16 -> 128 TTT sequences per task
    TASK_CAP_S = float(os.getenv("ARC_TASK_CAP_S", "1200"))  # per-task wall cap (train + decode)
    MIN_TASK_S = float(os.getenv("ARC_MIN_TASK_S", "0"))       # don't start a task with less time than this left
    FAIR_SHARE = os.getenv("ARC_FAIR_SHARE", "0") == "1"        # per-task cap = remaining time / remaining tasks per worker
    MIN_CAP_S = float(os.getenv("ARC_MIN_CAP_S", "600"))        # floor of the fair-share cap (TTT alone is ~3-5 min)

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=resolve_model_dir(),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        if MIN_TASK_S > 0 and end_time - time.time() < MIN_TASK_S:
            print(f"[Rank {rank}] {end_time - time.time():.0f}s left < {MIN_TASK_S:.0f}s; not starting {key}")
            break

        # Fair-share cap: instead of a fixed 1200 s, give this task its share of what is left so the tail of
        # the queue (largest tasks under SJF ordering) still gets TTT + at least one decode batch instead of nothing.
        task_cap = TASK_CAP_S
        if FAIR_SHARE:
            try:
                remaining = max(0, queue.qsize() - 4) + 1          # 4 sentinel Nones sit at the end of the queue
                share = (end_time - time.time()) / max(1, int(np.ceil(remaining / 4)))
                task_cap = float(min(TASK_CAP_S, max(MIN_CAP_S, share)))
            except Exception:
                task_cap = TASK_CAP_S
            print(f"[Rank {rank}] fair-share cap {task_cap:.0f}s for {key}")
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=TTT_AUG_N, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > task_cap or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=stable_seed_for_key(bk, os.getenv('ARC_AUG_SEED_OFFSET', 0)))
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")
        try:
            with open("/kaggle/working/phase_a_tasks.jsonl", "a") as f:
                f.write(json.dumps({"key": key, "rank": rank, "seconds": round(spend_time, 1), "cap": round(task_cap)}) + "\n")
        except Exception:
            pass

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    

    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


def task_cells(task):
    return sum(len(g["input"]) * len(g["input"][0]) + len(g.get("output", [[0]])) * len(g.get("output", [[0]])[0])
               for g in task["train"]) + sum(len(g["input"]) * len(g["input"][0]) for g in task["test"])


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    # Dev-mode task subset (non-rerun only). "all" = every task in the file.
    dev_tasks = os.getenv("ARC_DEV_TASKS", "0934a4d8,36a08778,981571dc,aa4ec2a5")
    dev_set = None if dev_tasks.strip().lower() == "all" else {k.strip() for k in dev_tasks.split(",") if k.strip()}

    keys = sorted(data.keys())
    if os.getenv("ARC_ORDER", "sorted") == "sjf":  # shortest-job-first: small tasks are the 4B's best value
        keys = sorted(keys, key=lambda k: (task_cells(data[k]), k))

    queue = mp.Manager().Queue()

    n_queued = 0
    for key in keys:
        if not rerun_mode and dev_set is not None and key not in dev_set:
            continue
        queue.put(key)
        n_queued += 1
    for _ in range(4):
        queue.put(None)
    print(f"[starter] queued {n_queued} tasks, end_time in {args.end_time - time.time():.0f}s")
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)


In [ ]:
import os, sys, json, time, signal, subprocess
CFG = json.load(open("/kaggle/working/arc_run_config.json"))
GRACE_S = 20 * 60          # Phase A workers only check the clock between tasks / DFS batches; cap the overshoot

if CFG["mode"] == "b_only":
    print("Phase A skipped (ARC_MODE=b_only)")
else:
    env = dict(os.environ)
    env.update({
        # identical to the original notebook's launch line (deterministic seeds + unsloth settings)
        "PYTHONHASHSEED": "260618",
        "ARC_AUG_SEED_OFFSET": "260618",
        "UNSLOTH_DISABLE_STATISTICS": "1",
        "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
        "OMP_NUM_THREADS": "12",
    })
    if CFG["mode"] == "hybrid":
        env.setdefault("ARC_MIN_TASK_S", "480")   # a task started with <8 min left cannot finish TTT + decode
        env.setdefault("ARC_ORDER", "sjf")        # small tasks first: highest solve-rate per GPU-minute
        env.setdefault("ARC_FAIR_SHARE", "1")     # adaptive per-task cap so the whole queue gets TTT
        env.setdefault("ARC_ORDER", "sjf")        # per-task results are seed-deterministic, so order only decides
                                                  # WHICH tasks finish before the cap: small tasks = best 4B value
    t = time.time()
    print(f"Phase A: 4B TTT on 4 x L4 until +{(CFG['phase_a_end'] - time.time())/3600:.2f} h")
    proc = subprocess.Popen([sys.executable, "starter.py", "--end-time", str(CFG["phase_a_end"])], env=env,
                            start_new_session=True)
    while proc.poll() is None:
        if CFG["mode"] == "hybrid" and time.time() > CFG["phase_a_end"] + GRACE_S:
            print("[Phase A] grace period over -> SIGKILL (finished tasks are already on disk)")
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except Exception:
                pass
            break
        time.sleep(20)
    try:
        proc.wait(timeout=60)
    except Exception:
        pass
    print(f"Phase A exit code {proc.returncode} after {(time.time() - t)/3600:.2f} h")


In [ ]:
import os, json, numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, hashable

CFG = json.load(open("/kaggle/working/arc_run_config.json"))
rerun_mode = CFG["rerun"]

data = ArcDataset.from_file(CFG["test_path"])
if not rerun_mode:
    data = data.load_replies(CFG["sol_path"])

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
os.makedirs(CFG["inference_outputs"], exist_ok=True)
decoder.load_decoded_results(CFG["inference_outputs"])

selected = decoder.run_selection_algo()           # kgmon ordering, exactly as the original
submission = data.get_submission(selected)

# Per-output candidate statistics: how many distinct grids the 4B produced and how many
# DFS beams support the top-2. Phase B uses this to decide where it may add/replace attempts.
stats = {}
for bk, guesses in decoder.decoded_results.items():
    support = {}
    for g in guesses.values():
        h = hashable(g["solution"])
        support[h] = support.get(h, 0) + 1
    ordered = [hashable(s) for s in selected.get(bk, [])]
    stats[bk] = {
        "n_distinct": len(support),
        "top_support": [support.get(h, 0) for h in ordered[:2]],
        "n_beams": len(guesses),
    }
# outputs the 4B never reached (queue exhausted by the time cap) simply have no entry -> Phase B treats as 0

# Full ordered candidate lists (up to 6 grids) - Phase B shows the top ones as hints and rewards agreement.
candidates = {}
for bk, guesses in decoder.decoded_results.items():
    support = {}
    for g in guesses.values():
        h = hashable(g["solution"])
        support[h] = support.get(h, 0) + 1
    ordered = selected.get(bk, [])[:6]
    candidates[bk] = {"cands": [np.asarray(c).tolist() for c in ordered],
                      "support": [support.get(hashable(c), 0) for c in ordered]}
with open("/kaggle/working/phase_a_candidates.json", "w") as f:
    json.dump(candidates, f)

with open("submission.json", "w") as f:            # ALWAYS a valid submission from here on
    json.dump(submission, f)
with open("/kaggle/working/submission_phase_a.json", "w") as f:
    json.dump(submission, f)
with open(CFG["phase_a_stats"], "w") as f:
    json.dump(stats, f)

n_out = sum(len(v) for v in submission.values())
n_covered = sum(1 for s in stats.values() if s["n_distinct"] >= 1)
n_two = sum(1 for s in stats.values() if s["n_distinct"] >= 2)
print(f"Phase A: {n_covered}/{n_out} outputs have >=1 candidate, {n_two} have >=2")

if not rerun_mode and decoder.decoded_results:
    decoder.benchmark_selection_algos()
    print("*** Phase A score:", data.validate_submission(json.load(open("submission.json"))))


In [ ]:
import os, sys, json, glob, time, shutil, subprocess
from pathlib import Path

CFG = json.load(open("/kaggle/working/arc_run_config.json"))
READY_FLAG = "/kaggle/working/phase_b/ready"
if os.path.exists(READY_FLAG):
    os.remove(READY_FLAG)

def skip(reason):
    print(f"Phase B SKIPPED: {reason}")
    print("submission.json from Phase A stands.")

def preflight():
    if CFG["mode"] == "a_only":
        return "ARC_MODE=a_only"
    if time.time() > CFG["phase_b_kill_at"] - 40 * 60:
        return f"only {(CFG['phase_b_kill_at'] - time.time())/60:.0f} min left; engine init alone needs more"
    if not Path(CFG["gpt_oss_dir"], "config.json").exists():
        return f"gpt-oss-120b not attached (looked at {CFG['gpt_oss_dir']}); Add Input -> Models -> danielhanchen/gpt-oss-120b"
    cfg = json.load(open(Path(CFG["gpt_oss_dir"], "config.json")))
    if cfg.get("model_type") != "gpt_oss":
        return f"model_type={cfg.get('model_type')} is not gpt_oss"
    n_gb = sum(p.stat().st_size for p in Path(CFG["gpt_oss_dir"]).glob("*.safetensors")) / 1e9
    if not (50 <= n_gb <= 80):
        return f"safetensors total {n_gb:.1f} GB is not the ~63 GB MXFP4 checkpoint (bf16 240 GB cannot fit 4 x L4)"
    if not CFG["wheelhouse"]:
        return "vLLM wheelhouse dataset not attached (needs vllm-0.11.0*.whl + deps; see build_wheelhouse_b.sh)"
    if sys.version_info[:2] != (3, 11):
        return f"python {CFG['python']} != 3.11 (wheelhouse is built for the pinned docker image with py3.11/torch 2.8.0)"
    try:
        import torch
        if not torch.__version__.startswith("2.8."):
            return f"torch {torch.__version__} != 2.8.x (vllm 0.11.0 is compiled against torch 2.8.0)"
        if torch.cuda.device_count() < 4:
            return f"{torch.cuda.device_count()} GPUs visible, TP=4 needs 4"
    except Exception as e:
        return f"torch probe failed: {e!r}"
    return None

reason = preflight()
if reason:
    skip(reason)
else:
    venv = CFG["venv_b"]
    shutil.rmtree(venv, ignore_errors=True)
    wheels = sorted(glob.glob(f"{CFG['wheelhouse']}/*.whl"))
    # torch / triton / nvidia-* come from the image (torch 2.8.0+cu128); installing them again wastes
    # ~5 GB of disk and minutes of time, and a second torch on sys.path is a classic ABI trap.
    EXCL = ("torch-", "triton-", "nvidia_", "nvidia-")
    wheels = [w for w in wheels if not Path(w).name.lower().startswith(EXCL)]
    print(f"installing {len(wheels)} wheels from {CFG['wheelhouse']} -> {venv}")
    t = time.time()
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--no-warn-script-location",
           "--find-links", CFG["wheelhouse"], "--target", venv] + wheels
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"pip rc={r.returncode} in {time.time()-t:.0f}s")
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        skip("pip install failed")
    else:
        env = dict(os.environ, PYTHONPATH=venv + os.pathsep + os.environ.get("PYTHONPATH", ""))
        probe = ("import torch, transformers, vllm, openai_harmony;"
                 "print('torch', torch.__version__, 'transformers', transformers.__version__, 'vllm', vllm.__version__);"
                 "from vllm import LLM, SamplingParams; print('vllm import ok')")
        r = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True, timeout=600)
        print(r.stdout[-2000:]); 
        if r.returncode != 0:
            print(r.stderr[-4000:])
            skip("vllm import probe failed (wheelhouse incompatible with this image)")
        else:
            Path(READY_FLAG).write_text("ok")
            print("Phase B environment ready")


In [ ]:
%%writefile solver_b.py
"""Phase B: gpt-oss-120b as a *verified* program-synthesis gap-filler for ARC-AGI-2.

Design (why this shape):
  * The 4B TTT line (Phase A) is the score engine (~28-34 LB). A 120B general model cannot do TTT
    on 4 x L4 and its direct-answer accuracy on ARC-AGI-2 is low single digits, so it must never
    *replace* the 4B - it fills what the 4B left empty or weak, and only with candidates that have
    external evidence: a Python program that reproduces EVERY training pair exactly.
  * vLLM TP=4, Marlin MXFP4 MoE (weight-only 4-bit on Ada), TRITON_ATTN (attention sinks).
    Weights ~16 GB/GPU, leaving ~3-4 GB/GPU for KV cache at gpu_memory_utilization=0.88.
  * Everything is time-boxed and checkpointed: results.json is rewritten after every chunk, so a
    hard kill at the deadline loses at most the in-flight chunk.

Rounds (each only if time remains):
  R1 fresh programs   : n=K1 samples / task, reasoning=medium
  R2 repair           : best partial program + failing-pair feedback, n=K2
  R3 direct grid      : outputs still without any candidate get direct answers (low confidence)
  R4 extra fresh      : reasoning=high for still-unsolved priority tasks
"""
import os, sys, json, time, re, math, random, signal, hashlib, subprocess, tempfile, traceback
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

# ─────────────────────────── config / logging ────────────────────────────
CFG = json.load(open(os.getenv("ARC_RUN_CONFIG", "/kaggle/working/arc_run_config.json")))
OUT_DIR = os.path.dirname(CFG["phase_b_results"])
os.makedirs(OUT_DIR, exist_ok=True)
STAGE_PATH = CFG["phase_b_stage"]
RESULTS_PATH = CFG["phase_b_results"]
T_START = time.time()
DEADLINE = float(os.getenv("ARC_B_DEADLINE", CFG["phase_b_kill_at"])) - 120   # own margin before the watchdog kill

K1 = int(os.getenv("ARC_B_K1", "6"))
K2 = int(os.getenv("ARC_B_K2", "2"))
K4 = int(os.getenv("ARC_B_K4", "4"))
EFFORT1 = os.getenv("ARC_B_EFFORT1", "medium")
EFFORT4 = os.getenv("ARC_B_EFFORT4", "high")
MAX_TOK_PROG = int(os.getenv("ARC_B_MAX_TOKENS", "6144"))
MAX_TOK_GRID = int(os.getenv("ARC_B_MAX_TOKENS_GRID", "4096"))
CHUNK_TASKS = int(os.getenv("ARC_B_CHUNK_TASKS", "8"))
TPS_GUESS = float(os.getenv("ARC_B_TPS_GUESS", "150"))       # aggregate decode tok/s until measured
EXEC_WORKERS = int(os.getenv("ARC_B_EXEC_WORKERS", str(min(32, (os.cpu_count() or 8)))))
PAIR_TIMEOUT_S = float(os.getenv("ARC_B_PAIR_TIMEOUT_S", "4"))
SEED = int(os.getenv("ARC_SEED", "0"))
MAX_PROMPT_CHARS = int(os.getenv("ARC_B_MAX_PROMPT_CHARS", "18000"))


def log(msg):
    print(f"[B +{(time.time()-T_START)/60:6.1f}m] {msg}", flush=True)


def stage(name):
    with open(STAGE_PATH, "a") as f:
        f.write(f"{time.time():.0f} {name}\n")


def time_left():
    return DEADLINE - time.time()


# ─────────────────────────── task loading / priority ─────────────────────
def load_tasks():
    tasks = json.load(open(CFG["test_path"]))
    if not CFG["rerun"]:
        dev = CFG.get("dev_tasks", "all")
        if dev.strip().lower() != "all":
            keep = {k.strip() for k in dev.split(",") if k.strip()}
            tasks = {k: v for k, v in tasks.items() if k in keep}
    return tasks


def load_phase_a_stats():
    try:
        return json.load(open(CFG["phase_a_stats"]))
    except Exception:
        return {}


def output_priority(stat):
    """0 = nothing from the 4B, 1 = one candidate, 2 = two candidates but attempt_2 is a single beam, 3 = solid."""
    if not stat or stat.get("n_distinct", 0) == 0:
        return 0
    if stat["n_distinct"] == 1:
        return 1
    ts = stat.get("top_support", [])
    if len(ts) >= 2 and ts[1] <= 1:
        return 2
    return 3


def build_targets(tasks, stats):
    targets = []
    for tid, task in tasks.items():
        prios = [output_priority(stats.get(f"{tid}_{i}")) for i in range(len(task["test"]))]
        p = 0 if CFG["mode"] == "b_only" else min(prios)
        targets.append((p, task_chars(task), tid))
    targets.sort()
    return [(tid, p) for p, _, tid in targets]


# ─────────────────────────── prompts ─────────────────────────────────────
def grid_to_text(g):
    return "\n".join("".join(str(int(c)) for c in row) for row in g)


def task_chars(task):
    return sum(len(grid_to_text(p["input"])) + len(grid_to_text(p.get("output", [[0]]))) for p in task["train"]) \
        + sum(len(grid_to_text(p["input"])) for p in task["test"])


SYSTEM_PROG = (
    "You are an expert Python programmer solving ARC-AGI puzzles.\n"
    "Each puzzle gives input->output grid examples that all follow ONE hidden transformation rule. "
    "Grids are shown as rows of digits; each digit is one cell's color (0 is usually background).\n"
    "Infer the rule from ALL examples, then implement it as Python.\n"
    "Your final answer must contain exactly one ```python code block that defines\n"
    "    def transform(grid: list[list[int]]) -> list[list[int]]\n"
    "Rules: pure function, deterministic, no input()/print()/file/network access; you may import numpy, "
    "math, itertools, collections, functools, copy, re. Return a NEW grid (list of lists of ints 0-9); "
    "the output size may differ from the input size. The function is executed on every example and "
    "must reproduce each example output exactly; then it is applied to the test inputs."
)

SYSTEM_GRID = (
    "You are an expert at ARC-AGI puzzles. Each puzzle gives input->output grid examples that all follow "
    "ONE hidden transformation rule. Grids are rows of digits (one digit per cell, 0 usually background). "
    "Infer the rule from ALL examples and apply it to the test input. "
    "Your final answer must contain exactly one ```text code block holding ONLY the output grid as rows of digits."
)


def examples_block(task):
    parts = []
    for i, p in enumerate(task["train"], 1):
        parts.append(f"Example {i}\nInput:\n{grid_to_text(p['input'])}\nOutput:\n{grid_to_text(p['output'])}\n")
    return "\n".join(parts)


def prompt_program(task, hints=None):
    """hints: {test_idx: [grid, ...]} proposals from the 4B (Phase A). They are shown as fallible
    proposals: a near-miss grid is a strong clue about the rule, but the program is still verified
    against every training pair, so a wrong hint can only waste samples, never poison the answer."""
    body = examples_block(task)
    for i, p in enumerate(task["test"], 1):
        body += f"\nTest input {i} (the program will be applied to this):\n{grid_to_text(p['input'])}\n"
        for h, g in enumerate((hints or {}).get(i - 1, [])[:2]):
            body += (f"Proposal {'AB'[h]} for test output {i} from a specialized grid model "
                     f"(often right, sometimes off by a few cells - verify against the examples):\n{grid_to_text(g)}\n")
    body += "\nWrite the program."
    return [{"role": "system", "content": SYSTEM_PROG}, {"role": "user", "content": body}]


def load_hints():
    """Phase A attempts (valid ones only) keyed by task -> test_idx -> [grids]."""
    if os.getenv("ARC_B_HINTS", "1") != "1":
        return {}
    try:
        sub = json.load(open(os.path.join(os.path.dirname(CFG["phase_a_stats"]), "submission_phase_a.json")))
        stats = load_phase_a_stats()
    except Exception:
        return {}
    hints = {}
    for tid, outs in sub.items():
        for j, att in enumerate(outs):
            st = stats.get(f"{tid}_{j}", {})
            grids = []
            if st.get("n_distinct", 0) >= 1 and valid_grid(att["attempt_1"]):
                grids.append(att["attempt_1"])
            if st.get("n_distinct", 0) >= 2 and valid_grid(att["attempt_2"]) and att["attempt_2"] not in grids:
                grids.append(att["attempt_2"])
            if grids:
                hints.setdefault(tid, {})[j] = grids
    return hints


def load_agreement_sets(hints):
    """task -> test_idx -> [grids] the 4B produced (up to 6, from phase_a_candidates.json); falls back to hints."""
    path = os.path.join(os.path.dirname(CFG["phase_a_stats"]), "phase_a_candidates.json")
    try:
        cands = json.load(open(path))
    except Exception:
        return hints
    out = {}
    for subkey, v in cands.items():
        tid, j = subkey.rsplit("_", 1)
        grids = [g for g in v.get("cands", []) if valid_grid(g)]
        if grids:
            out.setdefault(tid, {})[int(j)] = grids
    for tid, d in hints.items():                       # never lose an attempt that the candidate file lacks
        for j, gs in d.items():
            cur = out.setdefault(tid, {}).setdefault(j, [])
            cur += [g for g in gs if g not in cur]
    return out


def row_signatures(grids, min_len=5):
    sigs = set()
    for g in grids:
        for row in g:
            if len(row) >= min_len:
                sigs.add("".join(map(str, row)))
                sigs.add(",".join(map(str, row)))
    return sigs


def looks_like_lookup(code, task, hint_grids=()):
    """A program that embeds a training/test/proposal row verbatim is a lookup table, not a rule."""
    grids = ([p["output"] for p in task["train"]] + [p["input"] for p in task["train"]]
             + [p["input"] for p in task["test"]] + list(hint_grids))
    flat = re.sub(r"[\s\[\]()]", "", code)
    return any(sig in flat for sig in row_signatures(grids))


def prompt_repair(task, code, fail):
    body = examples_block(task)
    body += "\nA candidate program:\n```python\n" + code.strip() + "\n```\n"
    if fail.get("error"):
        body += f"\nIt raised an error on Example {fail['index']+1}: {fail['error']}\n"
    else:
        body += (f"\nIt is WRONG on Example {fail['index']+1}.\nExpected output:\n{grid_to_text(fail['expected'])}\n"
                 f"Program output:\n{grid_to_text(fail['got']) if fail.get('got') else '(invalid grid)'}\n")
    body += "\nFind the real rule and return the full corrected program in one ```python block."
    return [{"role": "system", "content": SYSTEM_PROG}, {"role": "user", "content": body}]


def prompt_grid(task, test_idx):
    body = examples_block(task)
    body += f"\nTest input:\n{grid_to_text(task['test'][test_idx]['input'])}\n\nGive the output grid."
    return [{"role": "system", "content": SYSTEM_GRID}, {"role": "user", "content": body}]


# ─────────────────────────── harmony output parsing ──────────────────────
FINAL_MARK = "<|channel|>final<|message|>"
END_MARKS = ("<|return|>", "<|end|>", "<|call|>", "<|endoftext|>")


def final_text(raw):
    """Return the 'final' channel content of a harmony completion (None if the model never got there)."""
    i = raw.rfind(FINAL_MARK)
    if i < 0:
        return None
    s = raw[i + len(FINAL_MARK):]
    for m in END_MARKS:
        j = s.find(m)
        if j >= 0:
            s = s[:j]
    return s.strip()


CODE_RE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)


def extract_code(raw):
    text = final_text(raw)
    src = text if text is not None else raw
    blocks = [b for b in CODE_RE.findall(src) if "def transform" in b]
    if not blocks and text is None:
        return None                       # ran out of tokens while reasoning
    if not blocks:
        k = src.rfind("def transform")
        if k < 0:
            return None
        blocks = [src[k:]]
    code = blocks[-1].strip()
    try:
        compile(code, "<candidate>", "exec")
    except SyntaxError:
        return None
    return code


GRID_RE = re.compile(r"```(?:text|plain)?\s*\n(.*?)```", re.S)


def extract_grid(raw):
    text = final_text(raw)
    if text is None:
        return None
    blocks = GRID_RE.findall(text) or [text]
    for b in reversed(blocks):
        rows = [ln.strip() for ln in b.strip().splitlines() if ln.strip()]
        rows = [r.replace(" ", "").replace(",", "") for r in rows]
        if rows and all(r.isdigit() for r in rows) and len({len(r) for r in rows}) == 1:
            g = [[int(c) for c in r] for r in rows]
            if valid_grid(g):
                return g
    return None


def valid_grid(g):
    return (isinstance(g, list) and 1 <= len(g) <= 30 and all(isinstance(r, list) for r in g)
            and 1 <= len(g[0]) <= 30 and all(len(r) == len(g[0]) for r in g)
            and all(isinstance(c, int) and 0 <= c <= 9 for r in g for c in r))


# ─────────────────────────── sandboxed execution ─────────────────────────
RUNNER_SRC = r'''
import sys, json, signal, resource, builtins, copy
resource.setrlimit(resource.RLIMIT_AS, (3 * 1024**3, 3 * 1024**3))
payload = json.load(sys.stdin)
code, inputs, per_pair = payload["code"], payload["inputs"], payload["timeout"]
ALLOWED = {"numpy", "math", "itertools", "collections", "functools", "copy", "re", "typing",
           "operator", "heapq", "bisect", "string", "statistics", "fractions", "random", "dataclasses", "enum"}
_imp = builtins.__import__
def safe_import(name, *a, **k):
    if name.split(".")[0] not in ALLOWED:
        raise ImportError("import of %r is not allowed" % name)
    return _imp(name, *a, **k)
BANNED = {"open", "exec", "eval", "compile", "input", "exit", "quit", "help", "breakpoint", "globals", "locals", "vars"}
safe_builtins = {k: getattr(builtins, k) for k in dir(builtins) if not k.startswith("_") and k not in BANNED}
safe_builtins["__import__"] = safe_import
g = {"__builtins__": safe_builtins, "__name__": "__candidate__"}
def normalize(out):
    try:
        import numpy as np
        if isinstance(out, np.ndarray):
            out = out.tolist()
    except Exception:
        pass
    if isinstance(out, tuple):
        out = list(out)
    if not isinstance(out, list) or not out:
        return None
    rows = []
    for r in out:
        if hasattr(r, "tolist"):
            r = r.tolist()
        if isinstance(r, tuple):
            r = list(r)
        if not isinstance(r, list) or not r:
            return None
        rows.append([int(c) for c in r])
    if not (1 <= len(rows) <= 30 and 1 <= len(rows[0]) <= 30):
        return None
    if any(len(r) != len(rows[0]) for r in rows) or any(c < 0 or c > 9 for r in rows for c in r):
        return None
    return rows
class Timeout(Exception):
    pass
def on_alarm(signum, frame):
    raise Timeout("time limit")
signal.signal(signal.SIGALRM, on_alarm)
res = []
try:
    signal.setitimer(signal.ITIMER_REAL, per_pair)
    exec(compile(code, "<candidate>", "exec"), g)
    signal.setitimer(signal.ITIMER_REAL, 0)
    fn = g["transform"]
except BaseException as e:
    print(json.dumps({"fatal": repr(e)[:300]}))
    sys.exit(0)
dead = None
for grid in inputs:
    if dead is not None:                      # a timeout/MemoryError makes the program useless: skip the rest
        res.append({"out": None, "error": dead}); continue
    try:
        signal.setitimer(signal.ITIMER_REAL, per_pair)
        out = fn(copy.deepcopy(grid))
        signal.setitimer(signal.ITIMER_REAL, 0)
        res.append({"out": normalize(out)})
    except BaseException as e:
        signal.setitimer(signal.ITIMER_REAL, 0)
        res.append({"out": None, "error": repr(e)[:300]})
        if isinstance(e, (Timeout, MemoryError)):
            dead = repr(e)[:300]
print(json.dumps({"results": res}))
'''
RUNNER_PATH = os.path.join(OUT_DIR, "candidate_runner.py")
with open(RUNNER_PATH, "w") as f:
    f.write(RUNNER_SRC)


def run_program(code, inputs, per_pair=PAIR_TIMEOUT_S):
    """Execute candidate code on `inputs` in an isolated subprocess. Returns list of {'out': grid|None, 'error'?}."""
    payload = json.dumps({"code": code, "inputs": inputs, "timeout": per_pair})
    budget = per_pair * (len(inputs) + 1) + 5
    try:
        r = subprocess.run([sys.executable, "-I", RUNNER_PATH], input=payload, capture_output=True,
                           text=True, timeout=budget, env={"PATH": os.environ.get("PATH", ""),
                                                            "PYTHONPATH": os.environ.get("ARC_B_RUNNER_PYTHONPATH", "")})
        line = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else "{}"
        data = json.loads(line)
        if "fatal" in data:
            return [{"out": None, "error": data["fatal"]} for _ in inputs]
        return data.get("results") or [{"out": None, "error": "no result"} for _ in inputs]
    except subprocess.TimeoutExpired:
        return [{"out": None, "error": "timeout"} for _ in inputs]
    except Exception as e:
        return [{"out": None, "error": f"runner: {e!r}"[:300]} for _ in inputs]


def evaluate_program(task, code):
    """Run on train+test inputs. Returns dict(n_pass, n_train, first_fail, test_outputs)."""
    inputs = [p["input"] for p in task["train"]] + [p["input"] for p in task["test"]]
    res = run_program(code, inputs)
    n_train = len(task["train"])
    n_pass, first_fail = 0, None
    for i in range(n_train):
        ok = res[i]["out"] == task["train"][i]["output"]
        n_pass += ok
        if not ok and first_fail is None:
            first_fail = {"index": i, "expected": task["train"][i]["output"], "got": res[i]["out"],
                          "error": res[i].get("error")}
    tests = [res[n_train + j]["out"] for j in range(len(task["test"]))]
    return {"n_pass": n_pass, "n_train": n_train, "first_fail": first_fail, "test_outputs": tests}


# ─────────────────────────── result store ────────────────────────────────
class Store:
    """results[tid] = {'programs': [...], 'outputs': {test_idx: [cand,...]}} - rewritten atomically."""

    def __init__(self):
        self.data = {}
        self.meta = {"started": T_START, "chunks": 0, "gen_tokens": 0, "gen_seconds": 0.0}
        if os.path.exists(RESULTS_PATH):
            try:
                old = json.load(open(RESULTS_PATH))
                self.data = old.get("tasks", {})
                self.meta.update(old.get("meta", {}))
            except Exception:
                pass

    def task(self, tid):
        return self.data.setdefault(tid, {"programs": [], "outputs": {}, "verified": 0})

    def add_program(self, tid, code, ev, origin, agree_with=None):
        """agree_with: {test_idx: [4B grids]} - a verified output that matches one is the strongest signal we have."""
        t = self.task(tid)
        t["programs"].append({"code": code, "n_pass": ev["n_pass"], "n_train": ev["n_train"],
                              "first_fail": ev["first_fail"], "test_outputs": ev["test_outputs"], "origin": origin})
        verified = ev["n_pass"] == ev["n_train"] and ev["n_train"] > 0
        t["verified"] += int(verified)
        for j, out in enumerate(ev["test_outputs"]):
            if out is None or not valid_grid(out):
                continue
            if verified:
                kind = "verified_agree" if out in (agree_with or {}).get(j, []) else "verified"
            else:
                kind = "partial" if ev["n_pass"] > 0 else "unverified"
            self.add_candidate(tid, j, out, kind, ev["n_pass"] / max(1, ev["n_train"]))

    def add_candidate(self, tid, j, grid, kind, score):
        lst = self.task(tid)["outputs"].setdefault(str(j), [])
        for c in lst:
            if c["grid"] == grid:
                c["count"] += 1
                c["n_verified"] = c.get("n_verified", 0) + (kind == "verified")
                c["kind"] = max(c["kind"], kind, key=KIND_RANK.get)
                c["score"] = max(c["score"], score)
                return
        lst.append({"grid": grid, "kind": kind, "count": 1, "n_verified": int(kind == "verified"), "score": score})

    def best_partial(self, tid):
        progs = [p for p in self.task(tid)["programs"] if p["n_pass"] < p["n_train"] and p["first_fail"]]
        if not progs:
            return None
        return max(progs, key=lambda p: (p["n_pass"], -len(p["code"])))

    def has_verified(self, tid):
        return self.task(tid)["verified"] > 0

    def n_candidates(self, tid, j):
        return len(self.task(tid)["outputs"].get(str(j), []))

    def has_evidence(self, tid, j):
        """True if some candidate for this output comes from a program that passes >=1 training pair."""
        return any(c["kind"] in ("verified", "verified_agree", "partial") for c in self.task(tid)["outputs"].get(str(j), []))

    def flush(self):
        tmp = RESULTS_PATH + ".tmp"
        with open(tmp, "w") as f:
            json.dump({"meta": self.meta, "tasks": self.data}, f)
        os.replace(tmp, RESULTS_PATH)


KIND_RANK = {"unverified": 0, "direct": 1, "partial": 2, "verified": 3, "verified_agree": 4}


# ─────────────────────────── LLM backend ─────────────────────────────────
class VllmBackend:
    def __init__(self):
        from vllm import LLM, SamplingParams
        self.SamplingParams = SamplingParams
        stage("engine_init_start")
        log(f"vLLM init: {CFG['gpt_oss_dir']} TP=4")
        kw = dict(
            model=CFG["gpt_oss_dir"],
            tokenizer=CFG["gpt_oss_dir"],
            tensor_parallel_size=4,
            dtype="bfloat16",
            max_model_len=int(os.getenv("ARC_B_MAX_MODEL_LEN", "16384")),
            gpu_memory_utilization=float(os.getenv("ARC_B_GPU_UTIL", "0.88")),
            max_num_seqs=int(os.getenv("ARC_B_MAX_NUM_SEQS", "32")),
            max_num_batched_tokens=int(os.getenv("ARC_B_MAX_BATCHED_TOKENS", "4096")),
            enforce_eager=os.getenv("ARC_B_EAGER", "0") == "1",
            enable_prefix_caching=True,
            distributed_executor_backend="mp",
            seed=SEED,
            trust_remote_code=False,
        )
        self.llm = LLM(**kw)
        stage("engine_ready")
        log("vLLM ready")

    def chat(self, conversations, n, max_tokens, effort, seed):
        sp = self.SamplingParams(n=n, temperature=1.0, top_p=1.0, max_tokens=max_tokens,
                                 skip_special_tokens=False, seed=seed)
        outs = self.llm.chat(conversations, sp, chat_template_kwargs={"reasoning_effort": effort}, use_tqdm=False)
        texts, ntok = [], 0
        for ro in outs:
            texts.append([o.text for o in ro.outputs])
            ntok += sum(len(o.token_ids) for o in ro.outputs)
        return texts, ntok


# ─────────────────────────── budgeted generation ─────────────────────────
class Budget:
    def __init__(self):
        self.tps = TPS_GUESS
        self.measured = False

    def worst_case_s(self, n_requests, max_tokens):
        # decode dominates; prefill of ~3-5k tokens per request is folded into the 1.3 factor
        return 1.3 * n_requests * max_tokens / self.tps

    def fits(self, n_requests, max_tokens):
        # actual completions run ~50% of max_tokens on average; be conservative until measured
        frac = 0.75 if self.measured else 1.0
        return time_left() > frac * self.worst_case_s(n_requests, max_tokens) + 30

    def update(self, ntok, seconds):
        if ntok > 0 and seconds > 0:
            tps = ntok / seconds
            self.tps = tps if not self.measured else 0.7 * self.tps + 0.3 * tps
            self.measured = True


def generate_chunk(backend, store, budget, items, n, max_tokens, effort, seed_base, on_result):
    """items: list of (key, conversation). Runs one batched generate + parallel verification."""
    t = time.time()
    convs = [c for _, c in items]
    texts, ntok = backend.chat(convs, n, max_tokens, effort, seed_base)
    gen_s = time.time() - t
    budget.update(ntok, gen_s)
    store.meta["gen_tokens"] += ntok
    store.meta["gen_seconds"] += gen_s
    store.meta["chunks"] += 1
    log(f"chunk: {len(items)} prompts x{n} -> {ntok} tokens in {gen_s:.0f}s ({ntok/max(gen_s,1e-6):.0f} tok/s)")
    t = time.time()
    jobs = [(key, raw) for (key, _), outs in zip(items, texts) for raw in outs]
    with ThreadPoolExecutor(max_workers=EXEC_WORKERS) as ex:
        list(ex.map(lambda kr: on_result(*kr), jobs))
    log(f"verified {len(jobs)} completions in {time.time()-t:.0f}s")
    store.flush()
    stage("alive")


def run_round(backend, store, budget, tasks, targets, make_conv, n, max_tokens, effort, origin, seed_off, agree=None):
    agree = agree or {}
    """targets: list of keys (task ids, or (tid, test_idx) for grid mode). Chunked, time-checked."""
    i = 0
    probe = not budget.measured and len(targets) > 0
    while i < len(targets) or probe:
        was_probe = probe
        if probe:                               # throughput probe: a few LOW-priority targets, 2 samples each
            chunk, nn, probe = targets[-4:], min(n, 2), False
        else:
            chunk, nn = targets[i:i + CHUNK_TASKS], n
        # shrink the chunk / samples until it fits the remaining time
        while chunk and not budget.fits(len(chunk) * nn, max_tokens):
            if nn > 1:
                nn -= 1
            else:
                chunk = chunk[:-1]
        if not chunk:
            log(f"{origin}: out of time with {len(targets) - i} targets left (tps={budget.tps:.0f})")
            return False
        items = [(k, make_conv(k)) for k in chunk]
        items = [(k, c) for k, c in items if c is not None and sum(len(m['content']) for m in c) <= MAX_PROMPT_CHARS]
        if items:
            def on_result(key, raw):
                tid = key if isinstance(key, str) else key[0]
                task = tasks[tid]
                if origin == "direct":
                    g = extract_grid(raw)
                    if g is not None:
                        store.add_candidate(tid, key[1], g, "direct", 0.0)
                    return
                code = extract_code(raw)
                if code is None:
                    return
                agree_with = agree.get(tid, {})
                if looks_like_lookup(code, task, [g for gs in agree_with.values() for g in gs]):
                    store.meta["lookup_rejected"] = store.meta.get("lookup_rejected", 0) + 1
                    return
                ev = evaluate_program(task, code)
                store.add_program(tid, code, ev, origin, agree_with=agree_with)
            seed = (SEED * 1000003 + seed_off + i + (7919 if was_probe else 0)) % (2**31)
            generate_chunk(backend, store, budget, items, nn, max_tokens, effort, seed, on_result)
        if not was_probe:
            i += len(chunk)
    return True


# ─────────────────────────── main ────────────────────────────────────────
def main():
    stage("start")
    tasks = load_tasks()
    stats = load_phase_a_stats()
    targets = build_targets(tasks, stats)
    by_p = Counter(p for _, p in targets)
    log(f"{len(tasks)} tasks; priority histogram {dict(sorted(by_p.items()))}; deadline in {time_left()/60:.0f} min")
    store = Store()
    budget = Budget()
    hints = load_hints()
    agree = load_agreement_sets(hints)
    log(f"hints from Phase A for {len(hints)} tasks; agreement sets for {len(agree)} tasks")

    if time_left() < 20 * 60:
        log("not enough time for engine init; exiting")
        store.flush(); return

    backend = FakeBackend() if os.getenv("ARC_B_FAKE_BACKEND") else VllmBackend()

    # R1: fresh programs for everything, priority order (P0 first; P3 only in b_only where all are P0)
    r1 = [tid for tid, p in targets if p <= (3 if CFG["mode"] == "b_only" else 2)]
    run_round(backend, store, budget, tasks, r1, lambda tid: prompt_program(tasks[tid], hints.get(tid)),
              K1, MAX_TOK_PROG, EFFORT1, "fresh", 11, agree)

    # R2: repair the best partial program of tasks that are still unverified
    r2 = [tid for tid in r1 if not store.has_verified(tid) and store.best_partial(tid)]
    def repair_conv(tid):
        bp = store.best_partial(tid)
        return prompt_repair(tasks[tid], bp["code"], bp["first_fail"]) if bp else None
    run_round(backend, store, budget, tasks, r2, repair_conv, K2, MAX_TOK_PROG, EFFORT1, "repair", 23, agree)

    # R3: direct grids for outputs the 4B left empty and no program with training evidence covers
    #     (a reasoned direct answer outranks the output of a program that fails every training pair)
    r3 = [(tid, j) for tid, p in targets for j in range(len(tasks[tid]["test"]))
          if output_priority(stats.get(f"{tid}_{j}")) == 0 and not store.has_evidence(tid, j)]
    run_round(backend, store, budget, tasks, r3, lambda key: prompt_grid(tasks[key[0]], key[1]),
              2, MAX_TOK_GRID, "medium", "direct", 37)

    # R4: more effort on the still-unsolved priority tasks while time remains
    r4 = [tid for tid, p in targets if p <= 1 and not store.has_verified(tid)]
    run_round(backend, store, budget, tasks, r4, lambda tid: prompt_program(tasks[tid], hints.get(tid)),
              K4, MAX_TOK_PROG, EFFORT4, "fresh_high", 53, agree)

    store.flush()
    nv = sum(1 for t in store.data.values() if t["verified"] > 0)
    na = sum(1 for t in store.data.values() if any(c["kind"] == "verified_agree" for v in t["outputs"].values() for c in v))
    log(f"done: {nv} tasks with a verified program ({na} agreeing with a 4B proposal), "
        f"{store.meta.get('lookup_rejected', 0)} lookup programs rejected; {store.meta['gen_tokens']} tokens in "
        f"{store.meta['gen_seconds']/60:.1f} min of generation")
    stage("done")


class FakeBackend:
    """Test double: returns canned harmony-formatted completions (used by the offline self-test)."""

    def __init__(self):
        stage("engine_init_start"); stage("engine_ready")
        self.script = json.load(open(os.environ["ARC_B_FAKE_BACKEND"]))

    def chat(self, conversations, n, max_tokens, effort, seed):
        texts = []
        for conv in conversations:
            user = conv[-1]["content"]
            key = "repair" if "candidate program" in user else ("grid" if "Give the output grid" in user else "prog")
            texts.append([self.script.get(key, "") for _ in range(n)])
        return texts, sum(len(t) for tt in texts for t in tt) // 4


if __name__ == "__main__":
    try:
        main()
    except Exception:
        traceback.print_exc()
        stage("crashed")
        sys.exit(1)


In [ ]:
import os, sys, json, time, signal, subprocess
from pathlib import Path

CFG = json.load(open("/kaggle/working/arc_run_config.json"))
READY_FLAG = "/kaggle/working/phase_b/ready"
STAGE = CFG["phase_b_stage"]
INIT_DEADLINE_S = 45 * 60      # engine must reach engine_ready within this
STALL_DEADLINE_S = 40 * 60     # after ready: no 'alive' heartbeat for this long -> kill

def read_stages():
    try:
        return [ln.split() for ln in open(STAGE).read().splitlines() if ln.strip()]
    except FileNotFoundError:
        return []

def kill_group(proc):
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except Exception:
        pass
    try:
        proc.wait(timeout=30)
    except Exception:
        pass

def launch(extra_env):
    env = dict(os.environ)
    env["PYTHONPATH"] = CFG["venv_b"] + os.pathsep + env.get("PYTHONPATH", "")
    env.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false",
                "VLLM_LOGGING_LEVEL": "INFO", "OMP_NUM_THREADS": "8", "PYTHONUNBUFFERED": "1"})
    env.update(extra_env)
    Path(STAGE).unlink(missing_ok=True)
    return subprocess.Popen([sys.executable, "solver_b.py"], env=env, start_new_session=True)

def supervise(proc):
    t_launch = time.time(); ready_at = None; last_alive = None
    while True:
        rc = proc.poll()
        if rc is not None:
            return rc, ready_at is not None
        now = time.time()
        st = read_stages()
        names = [s[1] for s in st]
        if ready_at is None and "engine_ready" in names:
            ready_at = float(st[names.index("engine_ready")][0]); last_alive = ready_at
            print(f"[watchdog] engine ready after {(ready_at - t_launch)/60:.1f} min")
        alive = [float(s[0]) for s in st if s[1] == "alive"]
        if alive:
            last_alive = alive[-1]
        if now >= CFG["phase_b_kill_at"]:
            print("[watchdog] hard deadline -> kill"); kill_group(proc); return "hard_deadline", ready_at is not None
        if ready_at is None and now - t_launch > INIT_DEADLINE_S:
            print("[watchdog] engine init deadline -> kill"); kill_group(proc); return "init_timeout", False
        if ready_at is not None and now - last_alive > STALL_DEADLINE_S:
            print("[watchdog] stall -> kill"); kill_group(proc); return "stall", True
        time.sleep(30)

if not os.path.exists(READY_FLAG):
    print("Phase B not ready -> skipped")
else:
    attempts = [{}, {"ARC_B_EAGER": "1", "ARC_B_GPU_UTIL": "0.92", "ARC_B_MAX_NUM_SEQS": "16"}]
    for k, extra in enumerate(attempts):
        left = CFG["phase_b_kill_at"] - time.time()
        if left < 35 * 60:
            print(f"[watchdog] {left/60:.0f} min left; not launching attempt {k+1}"); break
        print(f"[watchdog] launching Phase B attempt {k+1} {extra or ''} with {left/60:.0f} min window")
        proc = launch(extra)
        rc, got_ready = supervise(proc)
        print(f"[watchdog] attempt {k+1} ended: {rc}")
        if got_ready or rc == "hard_deadline":
            break               # results.json holds whatever finished; init succeeded so no point retrying
        # init failed (OOM / incompatibility): retry once in eager mode with more headroom
    try:
        r = json.load(open(CFG["phase_b_results"]))
        nv = sum(1 for t in r["tasks"].values() if t.get("verified", 0) > 0)
        print(f"Phase B results: {len(r['tasks'])} tasks touched, {nv} with a verified program, "
              f"{r['meta'].get('gen_tokens', 0)} generated tokens")
    except Exception as e:
        print(f"Phase B produced no results ({e})")


In [ ]:
%%writefile merge_b.py
"""Merge Phase A (4B TTT) and Phase B (gpt-oss-120b verified programs) into the final submission.

Policy (ARC_MERGE_POLICY, default 'agree'):
  fill      : Phase B only fills slots the 4B left empty ([[0]] placeholder).
  weak2     : fill + replace attempt_2 when the 4B's attempt_2 rests on a single DFS beam AND
              Phase B has a VERIFIED program output that differs from both attempts.
  agree     : weak2 + replace attempt_2 whenever a verified program AGREES with one of the 4B's own
              lower-ranked candidates (two independent solvers reached the same grid).
  verified2 : fill + replace attempt_2 whenever Phase B has a verified output not already in the attempts.
attempt_1 from the 4B is never touched. Phase B 'direct' / 'partial' candidates are used only to fill.
"""
import os, sys, json, numpy as np

PLACEHOLDER = [[0]]
KIND_RANK = {"unverified": 0, "direct": 1, "partial": 2, "verified": 3, "verified_agree": 4}
VERIFIED = ("verified", "verified_agree")


def rank_b(cands):
    return sorted(cands, key=lambda c: (KIND_RANK.get(c["kind"], 0), c["count"], c["score"]), reverse=True)


def merge(sub_a, stats_a, res_b, policy="agree"):
    sub = json.loads(json.dumps(sub_a))          # deep copy
    changes = {"filled": 0, "replaced2": 0}
    for tid, outs in sub.items():
        tb = (res_b.get("tasks", {}).get(tid) or {}).get("outputs", {})
        for j, att in enumerate(outs):
            cands = rank_b(tb.get(str(j), []))
            if not cands:
                continue
            st = stats_a.get(f"{tid}_{j}", {})
            a1, a2 = att["attempt_1"], att["attempt_2"]
            has1 = st.get("n_distinct", 0) >= 1 and a1 != PLACEHOLDER
            has2 = st.get("n_distinct", 0) >= 2 and a2 != PLACEHOLDER
            if not has1:                                   # 4B produced nothing: take best two from B
                att["attempt_1"] = cands[0]["grid"]
                rest = [c for c in cands[1:] if c["grid"] != cands[0]["grid"]]
                att["attempt_2"] = rest[0]["grid"] if rest else cands[0]["grid"]
                changes["filled"] += 1
                continue
            if not has2:                                   # 4B has one candidate: fill attempt_2
                rest = [c for c in cands if c["grid"] != a1]
                if rest:
                    att["attempt_2"] = rest[0]["grid"]; changes["filled"] += 1
                continue
            # 4B has two candidates: only a verified program that adds a NEW grid may replace attempt_2
            weak2 = len(st.get("top_support", [])) >= 2 and st["top_support"][1] <= 1
            vr = [c for c in cands if c["kind"] in VERIFIED and c["grid"] != a1 and c["grid"] != a2]
            if not vr:
                continue
            best = vr[0]
            if (policy == "verified2"
                    or (policy in ("weak2", "agree") and weak2)
                    or (policy == "agree" and best["kind"] == "verified_agree")):
                att["attempt_2"] = best["grid"]; changes["replaced2"] += 1
    return sub, changes


def score(sub, solutions):
    s = 0.0
    for k, v in solutions.items():
        for i, r in enumerate(v):
            if any(np.array_equal(r, sub[k][i][a]) for a in ("attempt_1", "attempt_2")):
                s += 1 / len(v)
    return s


def main():
    CFG = json.load(open("/kaggle/working/arc_run_config.json"))
    policy = os.getenv("ARC_MERGE_POLICY", "agree")
    sub_a = json.load(open("/kaggle/working/submission_phase_a.json"))
    stats_a = json.load(open(CFG["phase_a_stats"])) if os.path.exists(CFG["phase_a_stats"]) else {}
    try:
        res_b = json.load(open(CFG["phase_b_results"]))
    except Exception:
        res_b = {"tasks": {}}
    sub, ch = merge(sub_a, stats_a, res_b, policy)
    json.dump(sub, open("submission.json", "w"))
    print(f"merge policy={policy}: filled {ch['filled']} slots, replaced {ch['replaced2']} attempt_2 -> submission.json")
    if not CFG["rerun"] and CFG.get("sol_path"):
        sol = json.load(open(CFG["sol_path"]))
        sol = {k: v for k, v in sol.items() if k in sub}
        empty = {k: [{"attempt_1": PLACEHOLDER, "attempt_2": PLACEHOLDER} for _ in v] for k, v in sub_a.items()}
        only_b, _ = merge(empty, {}, res_b, policy)
        print(f"local score  A-only {score(sub_a, sol):.2f} | B-only {score(only_b, sol):.2f} | merged {score(sub, sol):.2f}  (of {len(sol)} tasks)")


if __name__ == "__main__":
    main()


In [ ]:
!python merge_b.py
import json, os; s = json.load(open("submission.json")); print(len(s), "tasks in submission.json,", os.path.getsize("submission.json")//1024, "KB")